In [0]:
# Widgets
dbutils.widgets.removeAll()
dbutils.widgets.dropdown("env", "dev", ["dev", "prod"], "Ambiente")
dbutils.widgets.text("process_date", "", "Fecha Proceso (DDMMYYYY)")
dbutils.widgets.text("archivo_base", "", "Nombre base del archivo ")
dbutils.widgets.text("tabla_destino", "", "tabla destino en bronze")
 

In [0]:
# Captura de parámetros
env           = dbutils.widgets.get("env")
process_date  = dbutils.widgets.get("process_date")
archivo_base  = dbutils.widgets.get("archivo_base").strip()
tabla_destino = dbutils.widgets.get("tabla_destino").strip()

# Construcción automática del archivo
archivo_fuente = f"{archivo_base}_{process_date}.csv"

# Ruta y tabla completa
catalog       = f"{env}_fraud"
path_landing  = f"/Volumes/{catalog}/raw/landing"
ruta_completa = f"{path_landing}/{archivo_fuente}"
tabla_full    = f"{catalog}.bronze.{tabla_destino}"

print(f"🚀 INICIO INGESTA BRONZE")
print(f"   Ambiente       : {env}")
print(f"   Fecha proceso  : {process_date}")
print(f"   Archivo base   : {archivo_base}")
print(f"   Archivo fuente : {archivo_fuente}")
print(f"   Tabla destino  : {tabla_full}\n")

In [0]:
from pyspark.sql.functions import lit
from datetime import datetime
import pytz

peru_tz = pytz.timezone('America/Lima')
ingestion_ts = datetime.now(peru_tz).strftime('%Y-%m-%d %H:%M:%S')

print(f"🕒 Timestamp Perú generado: {ingestion_ts}\n")

In [0]:
try:
    df = spark.read.csv(ruta_completa, header=True, inferSchema=True)
    
    # Auditoría
    df = df.withColumn("ingestion_timestamp", lit(ingestion_ts)) \
           .withColumn("source_file", lit(archivo_fuente)) \
           .withColumn("process_date", lit(process_date))
    
    # Escritura
    df.write.format("delta") \
      .mode("overwrite") \
      .option("overwriteSchema", "true") \
      .saveAsTable(tabla_full)
    
    registros = df.count()
    
    # ────────────────────────────────────────────────
    # RESUMEN EN FORMATO TABULAR SIMPLE (alineado manual)
    # ────────────────────────────────────────────────
    print("\n" + "═" * 85)
    print(" RESUMEN DE INGESTA ".center(85, "═"))
    print("═" * 85)
    
    print(f"{'Ambiente':<25} : {env}")
    print(f"{'Fecha proceso':<25} : {process_date}")
    print(f"{'Archivo fuente':<25} : {archivo_fuente}")
    print(f"{'Tabla destino':<25} : {tabla_full}")
    print(f"{'Registros ingestados':<25} : {registros:,}")
    print(f"{'Timestamp Perú':<25} : {ingestion_ts}")
    
    print("═" * 85 + "\n")
except Exception as e:
    print(" ERROR durante la ingesta:")
    print(str(e))

In [0]:
# ────────────────────────────────────────────────
# TOP 10 - Formato grilla nativa (horizontal, sin cortar texto)
# ────────────────────────────────────────────────
print(f"🔍 TOP 10 registros de {tabla_destino}")
print("   (formato tabla horizontal - palabras completas)\n")

# La forma más bonita y nativa en Databricks: Pandas + style
display(
    df.limit(2).toPandas().style
      .set_table_styles([
          {'selector': 'th', 'props': [('font-weight', 'bold'), 
                                       ('text-align', 'left'), 
                                       ('background-color', '#e6e6e6'),
                                       ('padding', '8px')]},
          {'selector': 'td', 'props': [('text-align', 'left'), 
                                       ('padding', '6px')]},
          {'selector': 'tr:nth-child(even)', 'props': [('background-color', '#f9f9f9')]},
          {'selector': 'tr:hover', 'props': [('background-color', '#f0f0f0')]}
      ])
      .set_properties(**{'text-align': 'left', 'white-space': 'normal', 'word-wrap': 'break-word'})
      .set_caption(f"TOP 10 - {tabla_destino}")
)
